# Comparison of the gradient boosting models
XGBoost, LightGBM, CatBoost

File1 - Data preparation and Division into sets

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
import sklearn
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

## Data preparation

In [ ]:
df = pd.read_csv('star_classification.csv')
df.head(10)

### Removal of the categorical variables

In [ ]:
df.columns

In [ ]:
df = df.drop(columns=['obj_ID','run_ID','rerun_ID','cam_col','field_ID','spec_obj_ID','plate','MJD','fiber_ID'])

In [ ]:
df.columns

### Dealing with  Nans and duplicates 

In this dataset lack of data or duplicates are errors.

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.info()

In [ ]:
print(df.duplicated().sum())

### Check the correction of target

In [ ]:
sns.histplot(df['class'])

Data is too unbalanced. 
Records describing galaxies must be reduced.

Method 1:
Data removed randomly

In [ ]:
galaxies = df[df['class'] == 'GALAXY']

In [ ]:
galaxies_to_keep = galaxies.sample(n=40000, random_state=19)

Replace all galaxies from the original dataset with galaxies_to_keep

In [ ]:
df2 = df.drop(df[df['class'] == 'GALAXY'].index)

In [ ]:
df_random_galaxies = pd.concat([galaxies_to_keep, df2])

In [ ]:
sns.histplot(df_random_galaxies['class'])

Method 2:
Removal of the most common data in features.

None of the outliers will be missed.

Personally preferred.

In [ ]:
features1 = ['u', 'g', 'r', 'i', 'z']
features2 = ['alpha', 'delta']
features3 = ['redshift']

In [ ]:
sns.boxplot(data=df[df['class'] == 'GALAXY'][features1])

In [ ]:
features = features1 + features2 + features3

In [ ]:
mean_val = df[df['class'] == 'GALAXY' ] [features].mean()

In [ ]:
df['distance_from_mean'] = abs(df[df['class'] == 'GALAXY' ][features] - mean_val).sum(axis=1)

In [ ]:
df_common_galaxies = df.sort_values(by =  'distance_from_mean', ascending= False).iloc[19445:]

In [ ]:
sns.histplot(df_common_galaxies['class'])

In [ ]:
df_common_galaxies.drop(columns = ['distance_from_mean'], inplace = True)

Check if sets have the same size

In [ ]:
df_common_galaxies[df_common_galaxies['class'] == 'GALAXY'].count()

In [ ]:
df_random_galaxies[df_random_galaxies['class'] == 'GALAXY'].count()

### Comparison of the data before and after removing galaxies

In [ ]:
sns.boxplot(df[df['class'] == 'GALAXY'][features1])

In [ ]:
sns.boxplot(df_common_galaxies[df_common_galaxies['class'] == 'GALAXY'][features1])

Removing the most common records did not significantly change the data - the approach can be used

In [ ]:
sns.boxplot(df_random_galaxies[df_random_galaxies['class'] == 'GALAXY'] [features1])

Both approaches did not change its representativnes

### Data validation

### Choose type of the data:

In [ ]:
df = df_common_galaxies

In [ ]:
#df = df_random_galaxies

### Correction for the STAR  & QSO type:

In [ ]:
sns.boxplot(df[df['class'] == 'STAR'][features1])

In [ ]:
sns.boxplot(df[df['class'] == 'QSO'][features1])

In [ ]:
df.sort_values(ascending= True, by = 'u')

u, g, z equals -9999 is an error, has to be removed

In [ ]:
df.drop( df[df['u'] < 0].index, inplace = True)

In [ ]:
df.sort_values(ascending= True, by = 'u')

In [ ]:
sns.boxplot(df[df['class'] == 'STAR'][features1])

Galaxies, Stars and QSO's data is now correct.

### Label Encoding

In [ ]:
le = LabelEncoder()
df['target'] = le.fit_transform(df['class'])
df = df.drop('class', axis = 1)

In [ ]:
df.tail(8)

### The last check - the dataset readiness

In [ ]:
df.isnull().sum()

In [ ]:
df.info()

In [ ]:
df.describe()

### Correlation

In [ ]:
corr = df.corr()
plt.figure(figsize=(10, 8)) 
sns.heatmap(corr, annot=True, fmt=".2f", linewidths=0.5)

### Division into sets

In [ ]:
X = df[['u','g','r','i','z','redshift','alpha','delta']]
Y = df[['target']]

In [ ]:
X_train_full, X_test, Y_train_full, Y_test = train_test_split(
    X, Y, test_size = 0.2, random_state = 42
)

In [ ]:
X_train, X_val, Y_train, Y_val = train_test_split(
    X_train_full, Y_train_full, test_size=0.25, random_state=42
)

In [ ]:
print(X.shape)
print(X_test.shape)
print(X_val.shape)
print(X_train.shape)

### Export of ready sets

In [ ]:
X_test.to_csv('sets/X_test.csv', index=False)
Y_test.to_csv('sets/Y_test.csv', index=False)

In [ ]:
X_val.to_csv('sets/X_val.csv', index=False)
Y_val.to_csv('sets/Y_val.csv', index=False)

In [ ]:
X_train.to_csv('sets/X_train.csv', index=False)
Y_train.to_csv('sets/Y_train.csv', index=False)